# 05 — Neurobehavioral Outcome Analysis

This notebook examines whether individual brain-network reorganization
following neurofeedback is associated with changes in clinical symptoms
of major depressive disorder.

The primary neurobehavioral analysis focuses on change in depression
severity and its relationship with the dominant functional-network
transition identified in the connectivity analysis.

Before calculating clinical change scores, the structure and
completeness of the clinical variables are audited at the participant
level.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

PARTICIPANTS_FILE = Path(
    r"C:\Neurofeedback_Data_Fall26\Imaging\Processed Rest1&2 Scans\participants.tsv"
)

participants = pd.read_csv(
    PARTICIPANTS_FILE,
    sep="\t"
)

FINAL_SUBJECTS = [
    "E3746", "E3799", "E3973", "E4051",
    "E4209", "E4253", "E4324", "E4350",
    "E4360", "E4484", "E4689", "E4697",
    "E4745", "E5215", "E5580", "E5586",
    "E5693", "E5694"
]

print("Metadata shape:", participants.shape)

print("\nClinical columns:")
print([
    c for c in participants.columns
    if c in [
        "MADRS", "MADRS_post",
        "HAMD", "HAMD_post",
        "PHQ", "PHQ_post",
        "post_days"
    ]
])

## 2 — Participant-Level Clinical Data Audit

The metadata contain multiple records per participant. Before deriving
pre-to-post symptom changes, all records corresponding to the final
imaging cohort are inspected to determine how baseline and post-treatment
clinical scores are represented.

In [ ]:
clinical_cols = [
    "Sub",
    "Exam",
    "Ses",
    "Group",
    "Datetime",
    "MADRS",
    "MADRS_post",
    "HAMD",
    "HAMD_post",
    "PHQ",
    "PHQ_post",
    "post_days"
]

clinical_rows = participants[
    participants["Exam"].astype(str).isin(FINAL_SUBJECTS)
][clinical_cols].copy()

clinical_rows = clinical_rows.sort_values(
    ["Exam", "Datetime"]
)

print("Rows belonging to final imaging cohort:", len(clinical_rows))
print("Unique Exam IDs:", clinical_rows["Exam"].nunique())

print("\nRows per participant:")
print(
    clinical_rows
    .groupby("Exam")
    .size()
)

display(clinical_rows)

## 3 — Clinical Records Across Participant Visits

The imaging-session rows contain baseline clinical measures but do not
contain the corresponding post-treatment scores. Therefore, all metadata
records associated with the same participant identifiers are examined
to locate the post-treatment clinical assessments.

In [ ]:
# Get Sub IDs corresponding to our 18 final imaging participants
final_sub_ids = (
    participants[
        participants["Exam"].astype(str).isin(FINAL_SUBJECTS)
    ]["Sub"]
    .dropna()
    .unique()
)

print("Final imaging participants:", len(FINAL_SUBJECTS))
print("Unique Sub IDs:", len(final_sub_ids))

# Retrieve ALL records for those same people
all_clinical_records = participants[
    participants["Sub"].isin(final_sub_ids)
][[
    "Sub",
    "Exam",
    "Ses",
    "Group",
    "Datetime",
    "MADRS",
    "MADRS_post",
    "HAMD",
    "HAMD_post",
    "PHQ",
    "PHQ_post",
    "post_days"
]].copy()

all_clinical_records = all_clinical_records.sort_values(
    ["Sub", "Datetime"]
)

print("\nTotal records across these participants:",
      len(all_clinical_records))

print("\nRows per Sub:")
print(
    all_clinical_records
    .groupby("Sub")
    .size()
)

print("\nNon-missing clinical values:")
print(
    all_clinical_records[
        [
            "MADRS", "MADRS_post",
            "HAMD", "HAMD_post",
            "PHQ", "PHQ_post",
            "post_days"
        ]
    ]
    .notna()
    .sum()
)

display(all_clinical_records)

## 4 — MADRS Clinical Outcome

MADRS was selected as the primary clinical outcome for the
neurobehavioral analysis. Baseline and post-treatment scores were
obtained from the first-session participant record.

Clinical improvement was defined as:

\[
\Delta \mathrm{MADRS}
=
\mathrm{MADRS}_{pre}
-
\mathrm{MADRS}_{post}
\]

such that positive values indicate reduced depressive symptom severity
and negative values indicate worsening symptoms.

In [ ]:
# First-session record for each final participant
clinical_first = (
    all_clinical_records
    .sort_values(["Sub", "Datetime"])
    .groupby("Sub", as_index=False)
    .first()
)

madrs = clinical_first[
    [
        "Sub",
        "Group",
        "MADRS",
        "MADRS_post",
        "post_days"
    ]
].copy()

madrs = madrs.rename(
    columns={
        "MADRS": "MADRS_pre"
    }
)

madrs["MADRS_change"] = (
    madrs["MADRS_pre"]
    - madrs["MADRS_post"]
)

print("Participants:", len(madrs))
print(
    "Complete MADRS pre/post:",
    madrs[
        ["MADRS_pre", "MADRS_post"]
    ].dropna().shape[0]
)

print(
    "Missing MADRS outcome:",
    madrs.loc[
        madrs["MADRS_change"].isna(),
        "Sub"
    ].tolist()
)

display(madrs)

## 5 — Linking Clinical and Imaging Participants

Clinical records are indexed by participant-level Sub identifiers,
whereas the imaging analysis uses Exam identifiers. The corresponding
imaging Exam ID is therefore retained for each participant to enable
participant-level integration of clinical and brain-network outcomes.

In [ ]:
# Map participant-level Sub IDs to the final imaging Exam IDs
id_map = participants[
    participants["Exam"].astype(str).isin(FINAL_SUBJECTS)
][["Sub", "Exam"]].copy()

id_map = id_map.drop_duplicates()

print("ID mappings:", len(id_map))
print("Unique Sub IDs:", id_map["Sub"].nunique())
print("Unique Exam IDs:", id_map["Exam"].nunique())

madrs_imaging = madrs.merge(
    id_map,
    on="Sub",
    how="left"
)

madrs_imaging = madrs_imaging[
    [
        "Sub",
        "Exam",
        "Group",
        "MADRS_pre",
        "MADRS_post",
        "MADRS_change",
        "post_days"
    ]
]

print(
    "\nMissing imaging IDs:",
    madrs_imaging["Exam"].isna().sum()
)

display(madrs_imaging)

## 6 — Dominant Brain-Network Transition

The 36 within- and between-network functional-connectivity changes
identified in the imaging analysis were summarized using principal
component analysis (PCA). The first principal component (PC1) represents
the dominant direction of participant-level network reorganization.

The PC1 orientation was standardized so that the mean loading was
positive. These participant-level transition scores were then linked
with MADRS symptom change for neurobehavioral analysis.

In [ ]:
from sklearn.decomposition import PCA

# Y_delta and delta_network_df were created in Notebook 04.
# First check whether they are available in this notebook.
print("Y_delta available:", "Y_delta" in globals())
print("delta_network_df available:", "delta_network_df" in globals())

In [ ]:
from pathlib import Path
import os

# Check likely output locations from the completed connectivity/prediction notebooks
candidate_dirs = [
    Path(r"C:\Users\rahin\outputs"),
    Path(r"C:\Users\rahin\OneDrive\Desktop\TU\Summer 2026\Brain fmri in R\neurofeedback-gnn"),
]

for d in candidate_dirs:
    print("\nDIR:", d)
    if d.exists():
        for p in d.rglob("*"):
            name = p.name.lower()
            if any(k in name for k in [
                "delta", "network", "connectivity", "fc", "pca", "prediction"
            ]):
                print(p)

In [ ]:
from itertools import combinations_with_replacement
from nilearn import datasets

# ---------------------------------
# Same imaging representation used
# in Notebook 04
# ---------------------------------

ZDELTA_DIR = Path(
    r"C:\Users\rahin\OneDrive\Desktop\TU\Summer 2026\Brain fmri in R\neurofeedback-gnn\outputs\delta_connectivity_216_fisherz"
)

SUBJECTS = FINAL_SUBJECTS

# Reconstruct Schaefer network labels
schaefer_info = datasets.fetch_atlas_schaefer_2018(
    n_rois=200,
    yeo_networks=7,
    resolution_mm=2
)

schaefer_labels = schaefer_info.labels[1:]

schaefer_labels = [
    label.decode("utf-8") if isinstance(label, bytes) else str(label)
    for label in schaefer_labels
]

schaefer_networks = [
    label.split("_")[2]
    for label in schaefer_labels
]

# 200 cortical + 16 subcortical ROIs
roi_networks = schaefer_networks + ["Subcortical"] * 16
roi_networks_array = np.array(roi_networks)

network_order = [
    "Vis",
    "SomMot",
    "DorsAttn",
    "SalVentAttn",
    "Limbic",
    "Cont",
    "Default",
    "Subcortical"
]

network_pairs = list(
    combinations_with_replacement(network_order, 2)
)

# ---------------------------------
# Reconstruct 36 Δ-network features
# ---------------------------------

delta_feature_rows = []

for subject in SUBJECTS:

    delta_file = (
        ZDELTA_DIR /
        f"{subject}_DeltaFC_Z_216.csv"
    )

    delta = pd.read_csv(
        delta_file,
        index_col=0
    ).to_numpy()

    row = {"Subject": subject}

    for net1, net2 in network_pairs:

        idx1 = np.where(
            roi_networks_array == net1
        )[0]

        idx2 = np.where(
            roi_networks_array == net2
        )[0]

        block = delta[
            np.ix_(idx1, idx2)
        ].copy()

        if net1 == net2:
            np.fill_diagonal(
                block,
                np.nan
            )

        feature_name = f"Delta_{net1}__{net2}"

        row[feature_name] = np.nanmean(block)

    delta_feature_rows.append(row)

delta_network_df = pd.DataFrame(delta_feature_rows)

print("Delta network matrix shape:",
      delta_network_df.shape)

print("Participants:",
      delta_network_df["Subject"].nunique())

print("Network-change features:",
      delta_network_df.shape[1] - 1)

print("Missing values:",
      delta_network_df.isna().sum().sum())

## 7 — Neurobehavioral Association

The dominant brain-network transition was summarized using the first
principal component of the 36 network-level Fisher-z connectivity
changes. Participant-level PC1 scores were then linked with change in
MADRS depression severity to test whether neural reorganization was
associated with clinical improvement.

In [ ]:
from sklearn.decomposition import PCA
from scipy.stats import pearsonr

# ---------------------------------
# Compute descriptive PC1 exactly
# from the 36 delta-network features
# ---------------------------------

Y_delta = (
    delta_network_df
    .drop(columns="Subject")
    .to_numpy()
)

pca = PCA()
pc_scores = pca.fit_transform(Y_delta)

# Orient PC1 so mean loading is positive,
# matching Notebook 04 interpretation
if pca.components_[0].mean() < 0:
    pc_scores[:, 0] *= -1
    pca.components_[0] *= -1

pc1_df = pd.DataFrame({
    "Exam": delta_network_df["Subject"],
    "PC1_transition": pc_scores[:, 0]
})

print(
    "PC1 variance explained:",
    round(pca.explained_variance_ratio_[0], 3)
)

# ---------------------------------
# Merge brain and clinical outcomes
# ---------------------------------

neurobehavioral_df = madrs_imaging.merge(
    pc1_df,
    on="Exam",
    how="left"
)

analysis_df = neurobehavioral_df.dropna(
    subset=["MADRS_change", "PC1_transition"]
).copy()

print("Neurobehavioral sample:", len(analysis_df))

# ---------------------------------
# Brain-clinical association
# ---------------------------------

r, p = pearsonr(
    analysis_df["PC1_transition"],
    analysis_df["MADRS_change"]
)

print(f"Pearson r = {r:.4f}")
print(f"p-value = {p:.4f}")

display(
    analysis_df[
        [
            "Sub",
            "Exam",
            "Group",
            "MADRS_change",
            "PC1_transition"
        ]
    ]
)

In [ ]:
from scipy.stats import pearsonr

for group in ["active", "sham"]:

    g = analysis_df[
        analysis_df["Group"] == group
    ]

    r_g, p_g = pearsonr(
        g["PC1_transition"],
        g["MADRS_change"]
    )

    print(f"\n{group.upper()}")
    print("n =", len(g))
    print(f"r = {r_g:.4f}")
    print(f"p = {p_g:.4f}")

print("\nMADRS change by group:")

print(
    analysis_df
    .groupby("Group")["MADRS_change"]
    .agg(["count", "mean", "std"])
)

## Sensitivity Analysis: Post-Neurofeedback Interval

Because post-intervention resting-state scans were acquired at different
intervals following neurofeedback, we tested whether the dominant
brain-network transition score (PC1) was associated with the number of
days between neurofeedback and post-intervention assessment.

In [ ]:
from scipy.stats import pearsonr

timing_df = neurobehavioral_df[
    ["Exam", "Group", "PC1_transition", "post_days"]
].dropna().copy()

r_days, p_days = pearsonr(
    timing_df["PC1_transition"],
    timing_df["post_days"]
)

print("Participants:", len(timing_df))
print(
    "post_days range:",
    timing_df["post_days"].min(),
    "to",
    timing_df["post_days"].max()
)

print(f"PC1 transition vs post_days: r = {r_days:.4f}, p = {p_days:.4f}")

print("\nData:")
print(timing_df.to_string(index=False))

In [ ]:
print("neurobehavioral_df columns:")
print(neurobehavioral_df.columns.tolist())

print("\nVariables containing 'pc':")
print([x for x in globals() if "pc" in x.lower()])

In [ ]:
from scipy.stats import pearsonr
import statsmodels.api as sm

adjust_df = neurobehavioral_df[
    ["PC1_transition", "MADRS_change", "post_days"]
].dropna().copy()

# Remove the effect of post_days from PC1
X_days = sm.add_constant(adjust_df["post_days"])
pc1_model = sm.OLS(
    adjust_df["PC1_transition"],
    X_days
).fit()
pc1_resid = pc1_model.resid

# Remove the effect of post_days from MADRS change
madrs_model = sm.OLS(
    adjust_df["MADRS_change"],
    X_days
).fit()
madrs_resid = madrs_model.resid

# Partial correlation
r_partial, p_partial = pearsonr(pc1_resid, madrs_resid)

print("Participants:", len(adjust_df))
print(
    f"PC1 vs MADRS change controlling for post_days: "
    f"r = {r_partial:.4f}, p = {p_partial:.4f}"
)

## Exploratory Analysis: Individual Network-Transition Fingerprints

Each participant's post-minus-pre functional connectivity change across
the 36 within- and between-network connections was treated as an
individual network-transition fingerprint.

We first quantified pairwise similarity between participants' transition
fingerprints using Pearson correlation. This analysis asks whether
participants exhibit similar patterns of distributed network
reorganization, independent of the overall magnitude of connectivity
change.

In [ ]:
import numpy as np
import pandas as pd

# 36-dimensional transition fingerprint for each participant
fingerprint_df = delta_network_df.copy()

subject_ids = fingerprint_df["Subject"].astype(str).tolist()

X_fingerprint = (
    fingerprint_df
    .drop(columns="Subject")
    .to_numpy(dtype=float)
)

print("Participants:", X_fingerprint.shape[0])
print("Transition features:", X_fingerprint.shape[1])
print("Missing values:", np.isnan(X_fingerprint).sum())

# Correlation between every pair of participants'
# 36-dimensional network-transition fingerprints
similarity_matrix = np.corrcoef(X_fingerprint)

similarity_df = pd.DataFrame(
    similarity_matrix,
    index=subject_ids,
    columns=subject_ids
)

print("\nSimilarity matrix shape:", similarity_df.shape)
print("\nFirst 5 x 5:")
print(similarity_df.iloc[:5, :5].round(3))

In [ ]:
from itertools import combinations

# Map imaging ID to intervention group
group_map = (
    neurobehavioral_df[["Exam", "Group"]]
    .drop_duplicates("Exam")
    .set_index("Exam")["Group"]
    .str.lower()
    .to_dict()
)

# Check that every fingerprint participant has a group
print("Subjects with group labels:",
      sum(s in group_map for s in subject_ids),
      "/", len(subject_ids))

pair_results = []

for s1, s2 in combinations(subject_ids, 2):

    sim = similarity_df.loc[s1, s2]

    g1 = group_map[s1]
    g2 = group_map[s2]

    if g1 == "active" and g2 == "active":
        pair_type = "Active-Active"
    elif g1 == "sham" and g2 == "sham":
        pair_type = "Sham-Sham"
    else:
        pair_type = "Active-Sham"

    pair_results.append({
        "Subject1": s1,
        "Subject2": s2,
        "Similarity": sim,
        "PairType": pair_type
    })

pair_df = pd.DataFrame(pair_results)

print("\nTotal unique participant pairs:", len(pair_df))

print("\nNumber of pairs:")
print(pair_df["PairType"].value_counts())

print("\nFingerprint similarity:")
print(
    pair_df.groupby("PairType")["Similarity"]
    .agg(["count", "mean", "std", "median"])
    .round(4)
)

In [ ]:
import numpy as np

rng = np.random.default_rng(42)
n_perm = 10000

subjects = np.array(subject_ids)

# Original group labels
labels = np.array([group_map[s] for s in subjects])

def similarity_stat(labels_now):
    aa = []
    cross = []

    for i in range(len(subjects)):
        for j in range(i + 1, len(subjects)):

            sim = similarity_matrix[i, j]

            if labels_now[i] == "active" and labels_now[j] == "active":
                aa.append(sim)

            elif labels_now[i] != labels_now[j]:
                cross.append(sim)

    return np.mean(aa) - np.mean(cross)


observed_stat = similarity_stat(labels)

null_stats = np.empty(n_perm)

for p in range(n_perm):
    permuted_labels = rng.permutation(labels)
    null_stats[p] = similarity_stat(permuted_labels)

# One-sided permutation p-value:
# Is Active-Active similarity greater than Active-Sham similarity?
p_perm = (
    np.sum(null_stats >= observed_stat) + 1
) / (n_perm + 1)

print(f"Observed AA - AS similarity difference: {observed_stat:.4f}")
print(f"Permutation null mean: {null_stats.mean():.4f}")
print(f"Permutation null SD: {null_stats.std():.4f}")
print(f"Permutation p-value: {p_perm:.4f}")

## Robustness Check: Cosine Similarity

To assess whether the active-group convergence result depended on the
choice of Pearson correlation as the fingerprint-similarity metric,
we repeated the analysis using cosine similarity between the
36-dimensional network-transition fingerprints.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Compute cosine similarity between all participant fingerprints
cosine_matrix = cosine_similarity(X_fingerprint)

cosine_df = pd.DataFrame(
    cosine_matrix,
    index=subject_ids,
    columns=subject_ids
)

# Descriptive pairwise similarities
cos_pair_results = []

for s1, s2 in combinations(subject_ids, 2):

    sim = cosine_df.loc[s1, s2]

    g1 = group_map[s1]
    g2 = group_map[s2]

    if g1 == "active" and g2 == "active":
        pair_type = "Active-Active"
    elif g1 == "sham" and g2 == "sham":
        pair_type = "Sham-Sham"
    else:
        pair_type = "Active-Sham"

    cos_pair_results.append({
        "Subject1": s1,
        "Subject2": s2,
        "Similarity": sim,
        "PairType": pair_type
    })

cos_pair_df = pd.DataFrame(cos_pair_results)

print("Cosine similarity summary:")
print(
    cos_pair_df.groupby("PairType")["Similarity"]
    .agg(["count", "mean", "std", "median"])
    .round(4)
)

# Same AA - AS statistic
def cosine_stat(labels_now):
    aa = []
    cross = []

    for i in range(len(subjects)):
        for j in range(i + 1, len(subjects)):

            sim = cosine_matrix[i, j]

            if labels_now[i] == "active" and labels_now[j] == "active":
                aa.append(sim)

            elif labels_now[i] != labels_now[j]:
                cross.append(sim)

    return np.mean(aa) - np.mean(cross)

observed_cos = cosine_stat(labels)

rng = np.random.default_rng(42)
n_perm = 10000
null_cos = np.empty(n_perm)

for p in range(n_perm):
    permuted_labels = rng.permutation(labels)
    null_cos[p] = cosine_stat(permuted_labels)

p_cos = (
    np.sum(null_cos >= observed_cos) + 1
) / (n_perm + 1)

print("\nObserved cosine AA - AS difference:", round(observed_cos, 4))
print("Permutation null mean:", round(null_cos.mean(), 4))
print("Permutation null SD:", round(null_cos.std(), 4))
print("Permutation p-value:", round(p_cos, 4))

In [ ]:
print("neurobehavioral_df columns:")
print(neurobehavioral_df.columns.tolist())

print("\nneurobehavioral_df:")
print(neurobehavioral_df.to_string(index=False))

In [ ]:
# =======================================================
# FIGURE 7
# Neurobehavioral association: PC1 vs MADRS improvement
# =======================================================

import numpy as np
import matplotlib.pyplot as plt

plot_df = neurobehavioral_df.dropna(
    subset=["MADRS_change", "PC1_transition"]
).copy()

active_df = plot_df[plot_df["Group"] == "active"]
sham_df   = plot_df[plot_df["Group"] == "sham"]

fig, ax = plt.subplots(figsize=(6.5, 5.5))

# Individual participants
ax.scatter(
    active_df["PC1_transition"],
    active_df["MADRS_change"],
    s=55,
    label="Active"
)

ax.scatter(
    sham_df["PC1_transition"],
    sham_df["MADRS_change"],
    s=55,
    marker="s",
    label="Sham"
)

# Overall linear trend
x = plot_df["PC1_transition"].to_numpy()
y = plot_df["MADRS_change"].to_numpy()

coef = np.polyfit(x, y, 1)
x_line = np.linspace(x.min(), x.max(), 100)
y_line = np.polyval(coef, x_line)

ax.plot(
    x_line,
    y_line,
    linestyle="--",
    linewidth=1.5
)

ax.axhline(
    0,
    linewidth=1,
    linestyle=":"
)

ax.set_xlabel("Dominant brain-network transition score (PC1)")
ax.set_ylabel("MADRS improvement (Pre − Post)")

ax.set_title(
    "Neurobehavioral Association"
)

ax.text(
    0.05,
    0.95,
    r"$n=17$" + "\n" +
    r"$r=-0.305$" + "\n" +
    r"$p=0.234$",
    transform=ax.transAxes,
    va="top"
)

ax.legend(
    frameon=False
)

plt.tight_layout()

plt.savefig(
    "Fig7_Neurobehavioral_Association.png",
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    "Fig7_Neurobehavioral_Association.svg",
    bbox_inches="tight"
)

plt.show()